<!--
Projeto : Agente Especialista em GRC - RAG + LLM local sobre o COSO ERM 2017
Arquivo : llama/evaluateRAG.ipynb
-->
# Avaliação do recuperador (RAG)

**Universidade Presbiteriana Mackenzie** &ndash; Faculdade de Computação e Informática
Sistemas de Informação &ndash; Inteligência Artificial (7ºK, noite) &ndash; Prof. Dr. Leandro Zerbinatti

| Integrante | RA | E-mail |
|---|---|---|
| Guilherme Martins Silva | 10417140 | 10417140@mackenzista.com.br |
| Enzo Ribeiro | 10418262 | 10418262@mackenzista.com.br |
| Pedro Nogueira Ribeiro | 10324924 | 10324924@mackenzista.com.br |

**Síntese:** mede Hit Rate, MRR e Precision@1 do pipeline de recuperação do agente (MMR com k=20 no Chroma + re-ranking com BAAI/bge-reranker-v2-m3, top 6) sobre 16 perguntas com trecho de referência, variando o threshold de similaridade de 0,75 a 0,99. A comparação com baselines está em `experimentos_N2.ipynb`.

**Histórico de alterações**

| Data | Autor | Descrição |
|---|---|---|
| 2026-02-26 | Guilherme Martins Silva | Versão inicial: conjunto de 16 perguntas, pipeline e gráficos. |
| 2026-11-15 | Enzo Ribeiro | Cabeçalho padronizado; conjunto de perguntas movido para `llama/eval_dataset.py`; `pip install` substituído pelo `requirements.txt`; figura salva em `resultados/`; notebook reexecutado com as saídas salvas. |

> Dependências: `pip install -r requirements.txt` (na raiz do repositório). Execute este notebook a partir da pasta `llama/`.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))   # raiz do repositório -> pacote "llama"

from llama.EvaluationRAG import RAGEvaluator, EvaluationSample
from llama.eval_dataset import load_dataset
from llama.RAG.rank_context import RankContext
from llama.RAG.BuildVectorStore import PDFVectorStore

# 16 perguntas em português + trecho de referência (gabarito) do COSO ERM 2017
dataset = load_dataset("pt")
for i, s in enumerate(dataset, 1):
    print(f"{i:2d}. {s.query}")

In [ ]:
def build_pipeline():

    # 1. Carrega o banco e o reranker uma única vez (na inicialização)
    rag_support = PDFVectorStore()   # índice persistido em llama/RAG/db
    chroma = rag_support._load_db()

    reranker = RankContext(
        model="BAAI/bge-reranker-v2-m3",
        top_n=6
    )

    def pipeline(query: str, k: int, threshold=0.75):
        # STAGE 1: Busca MMR (Diversidade)
        # k aqui deve ser maior que o top_n do reranker para ele ter o que filtrar.
        # Se você quer 6 no final, peça 20 ou 30 para o MMR.
        initial_docs = chroma.max_marginal_relevance_search(
            query, 
            k=k,        # Quantidade para o Re-ranker analisar
            fetch_k=50,  # Quantidade que o Chroma analisa internamente
            lambda_mult=0.5
        )

        # STAGE 2: Re-ranking (Ordenação de Relevância)
        # O Re-ranker vai pegar esses 20 diversos e colocar os 6 melhores no topo
        ranked_docs = reranker.call_rank(query, initial_docs)

        # STAGE 3: Filtro Opcional de Threshold (se necessário)
        # Nota: Como o MMR não retorna score facilmente, o threshold aqui 
        # costuma ser aplicado no score do Re-ranker se o modelo permitir.
        # Por enquanto, vamos retornar os top_n ordenados:
        
        return [d.page_content for d in ranked_docs]

    return pipeline

In [ ]:
# 1. Constrói o pipeline do agente (MMR + re-ranking)
pipeline = build_pipeline()

# 2. Instancia o Evaluator
evaluator = RAGEvaluator(
    pipeline_fn=pipeline,
    dataset=dataset,
    model_name="BAAI/bge-m3"
)

# 3. Roda a avaliação
# k=20 é o que o retriever (Chroma) busca antes do Reranker filtrar para top_n=6
# similarity_threshold=0.70
df = evaluator.evaluate(k=20, similarity_threshold=0.75)

In [ ]:
import matplotlib.pyplot as plt

# 1. Criamos a figura com 3 subplots verticais
fig, axs = plt.subplots(3, 1, figsize=(10, 15), sharex=True)

# 2. Atualizamos a lista de métricas para os novos nomes das colunas
# (coluna_no_df, nome_exibição, marcador, cor, offset_texto)
metrics = [
    ('hit_rate', 'Hit Rate (Success @ K)', 'o', 'blue', (0, 7)),
    ('mrr', 'MRR (Mean Reciprocal Rank)', 'D', 'purple', (0, 7)),
    ('precision_at_1', 'Precision @ 1', 's', 'darkred', (0, 7))
]

for ax, (col, label, marker, color, offset) in zip(axs, metrics):
    # Plotagem da linha
    ax.plot(df['similarity_threshold'], df[col], 
            marker=marker, label=label, color=color, linewidth=2)

    # Adicionando anotações de dados (os valores numéricos sobre os pontos)
    for i in range(len(df)):
        val = df[col].iloc[i]
        threshold = df['similarity_threshold'].iloc[i]
        ax.annotate(f'{val:.2f}', 
                    (threshold, val), 
                    textcoords="offset points", 
                    xytext=offset, 
                    ha='center', 
                    fontsize=9,
                    fontweight='bold')

    # Configurações estéticas de cada subplot
    ax.set_ylabel('Score (0 a 1)')
    ax.set_title(f'{label} vs. Similarity Threshold', fontsize=12, pad=10)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_ylim(-0.05, 1.05) # Garante que a escala seja sempre de 0 a 1
    ax.legend(loc='upper right')

# Eixo X comum
axs[-1].set_xlabel('Similarity Threshold (Corte de Similaridade)')

# Ajuste fino para evitar sobreposição
plt.tight_layout()
os.makedirs("resultados", exist_ok=True)
plt.savefig("resultados/fig_avaliacao_retriever.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Ponto de operação escolhido (melhor equilíbrio entre cobertura e exigência semântica)
df[df.similarity_threshold.round(2) == 0.77]

In [ ]:
df